# 💻 Developer Burnout — Complete ML Pipeline
### Dataset: developer_burnout_dataset.csv
### Tasks: Regression (stress_level) + Classification (burnout_level)
---

## 📦 Step 1 — Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Splitting & Tuning
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV, RandomizedSearchCV

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    recall_score, precision_score, f1_score,
    matthews_corrcoef, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries imported!')

## 📁 Step 2 — Load Data

In [3]:
# Update path to your dataset location
df = pd.read_csv('./Dataset/developer_burnout.csv')
print('Shape:', df.shape)
df.head()

Shape: (7000, 12)


,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
0,26.0,12.0,10.33,4.45,2.0,11.0,4.0,1.0,15.07,0.14,55.96,Medium
1,39.0,10.0,8.62,5.77,5.0,15.0,11.0,5.0,13.25,0.54,82.22,High
2,34.0,13.0,NaN,4.03,5.0,2.0,18.0,9.0,11.18,1.54,61.77,Medium
3,30.0,1.0,6.85,6.47,2.0,15.0,26.0,1.0,11.14,0.96,54.98,Medium
4,27.0,7.0,4.24,5.80,NaN,9.0,17.0,7.0,8.05,0.36,27.90,Low


## 🔎 Step 3 — Data Exploration

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print('\nNull Values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nDescriptive Stats:')
df.describe()

In [ ]:
# Burnout Level Distribution
print('burnout_level unique values:', df['burnout_level'].unique())
print(df['burnout_level'].value_counts())

In [ ]:
# Outlier Detection — IQR Method
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
outlier_counts = []
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_counts.append([col, n, round(n/len(df)*100, 2)])

outlier_df = pd.DataFrame(outlier_counts, columns=[
                          'Feature', 'Outliers', 'Percentage'])
print(outlier_df.sort_values('Outliers', ascending=False))

## 📊 Step 4 — Visualization (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Burnout Level Count
sns.countplot(data=df, x='burnout_level', hue='burnout_level',
              palette='Set2', legend=False, ax=axes[0, 0])
axes[0, 0].set_title('Burnout Level Distribution')

# 2. Pie Chart
rc = df['burnout_level'].value_counts()
axes[0, 1].pie(rc, labels=rc.index, autopct='%1.1f%%')
axes[0, 1].set_title('Burnout Level Proportion')

# 3. Correlation Heatmap
num_df = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(num_df.corr(), annot=True,
            cmap='coolwarm', fmt='.2f', ax=axes[0, 2])
axes[0, 2].set_title('Correlation Matrix')

# 4. Work Hours vs Stress Level
sns.regplot(data=df, x='daily_work_hours', y='stress_level',
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'}, ax=axes[1, 0])
axes[1, 0].set_title('Work Hours vs Stress Level')

# 5. Sleep vs Stress
sns.scatterplot(data=df, x='sleep_hours', y='stress_level',
                hue='burnout_level', palette='Set1', ax=axes[1, 1])
axes[1, 1].set_title('Sleep Hours vs Stress Level')

# 6. Caffeine vs Stress
sns.boxplot(data=df, x='burnout_level', y='caffeine_intake',
            palette='Accent', ax=axes[1, 2])
axes[1, 2].set_title('Caffeine Intake by Burnout Level')

plt.tight_layout()
plt.show()

In [ ]:
# Violin Plots — Developer-specific features
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
plots = [
    ('burnout_level', 'stress_level',     'Stress by Burnout Level'),
    ('burnout_level', 'daily_work_hours', 'Work Hours by Burnout'),
    ('burnout_level', 'sleep_hours',      'Sleep by Burnout Level'),
    ('burnout_level', 'commits_per_day',  'Commits by Burnout'),
    ('burnout_level', 'bugs_per_day',     'Bugs by Burnout Level'),
]
for ax, (x, y, title) in zip(axes, plots):
    sns.violinplot(data=df, x=x, y=y, palette='Set2', inner='quartile', ax=ax)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Developer-specific analysis
print('Average Commits per Burnout Level:')
print(df.groupby('burnout_level')['commits_per_day'].mean().round(2))
print('\nAverage Bugs per Burnout Level:')
print(df.groupby('burnout_level')['bugs_per_day'].mean().round(2))
print('\nAverage Experience per Burnout Level:')
print(df.groupby('burnout_level')['experience_years'].mean().round(2))

## ⚙️ Step 5 — Feature Engineering

In [ ]:
df_feat = df.copy()

# Developer-specific engineered features
df_feat['bugs_per_commit'] = np.where(df_feat['commits_per_day'] != 0,
                                      np.round(df_feat['bugs_per_day'] / df_feat['commits_per_day'], 2), np.nan)
df_feat['work_sleep_ratio'] = np.round(
    df_feat['daily_work_hours'] / df_feat['sleep_hours'].replace(0, np.nan), 2)
df_feat['screen_sleep_ratio'] = np.round(
    df_feat['screen_time'] / df_feat['sleep_hours'].replace(0, np.nan), 2)
df_feat['sleep_deficit'] = np.round(
    np.maximum(0, 8 - df_feat['sleep_hours']), 2)
df_feat['total_digital_load'] = df_feat['daily_work_hours'] + \
    df_feat['screen_time'] + df_feat['meetings_per_day']

print('New features: bugs_per_commit, work_sleep_ratio, screen_sleep_ratio, sleep_deficit, total_digital_load')
df_feat[['bugs_per_commit', 'work_sleep_ratio', 'screen_sleep_ratio',
         'sleep_deficit', 'total_digital_load']].describe()

## 🧹 Step 6 — Preprocessing

In [ ]:
df_clean = df_feat.copy()

# Encode target
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_clean['burnout_level'] = df_clean['burnout_level'].map(risk_map)

# Drop NaN rows created by feature engineering
df_clean = df_clean.dropna()

print('Shape after cleaning:', df_clean.shape)
print('Encoding done!')
print(df_clean['burnout_level'].value_counts())

---
# 🔢 PART A — REGRESSION (Predict stress_level)
---

In [ ]:
X_reg = df_clean.drop(columns=['stress_level', 'burnout_level'])
y_reg = df_clean['stress_level']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

scaler_r = RobustScaler()
X_tr_sc = scaler_r.fit_transform(X_tr)
X_te_sc = scaler_r.transform(X_te)

reg_records = []

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_tr_sc, y_tr)
lr_pred = lr.predict(X_te_sc)
lr_r2 = r2_score(y_te, lr_pred)
reg_records.append({'Model': 'Linear Regression', 'R2': round(
    lr_r2, 4), 'MAE': round(mean_absolute_error(y_te, lr_pred), 4)})
print(f'Linear Regression R2: {lr_r2:.4f}')

# 2. Gradient Boosting Regressor
gb_r = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
gb_r.fit(X_tr_sc, y_tr)
gb_pred = gb_r.predict(X_te_sc)
gb_r2 = r2_score(y_te, gb_pred)
reg_records.append({'Model': 'GB Regressor', 'R2': round(
    gb_r2, 4), 'MAE': round(mean_absolute_error(y_te, gb_pred), 4)})
print(f'GB Regressor R2     : {gb_r2:.4f}')

# 3. XGBoost Regressor
xgb_r = XGBRegressor(n_estimators=500, learning_rate=0.01,
                     max_depth=3, reg_lambda=1, random_state=42)
xgb_r.fit(X_tr_sc, y_tr)
xgb_pred = xgb_r.predict(X_te_sc)
xgb_r2 = r2_score(y_te, xgb_pred)
reg_records.append({'Model': 'XGBoost Regressor', 'R2': round(
    xgb_r2, 4), 'MAE': round(mean_absolute_error(y_te, xgb_pred), 4)})
print(f'XGBoost Regressor R2: {xgb_r2:.4f}')

In [ ]:
reg_df = pd.DataFrame(reg_records).sort_values('R2', ascending=False)
print(reg_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=reg_df, x='Model', y='R2', palette='Greens_d')
plt.title('Developer Burnout — Regression R2 Scores')
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Regression → Classification (stress_level → burnout_level)
stress_min = df_clean['stress_level'].min()
stress_max = df_clean['stress_level'].max()
low_thresh = stress_min + (stress_max - stress_min) * 0.33
high_thresh = stress_min + (stress_max - stress_min) * 0.66

print(f'Stress range: {stress_min:.2f} to {stress_max:.2f}')
print(f'Low threshold  : < {low_thresh:.2f}')
print(f'Medium threshold: {low_thresh:.2f} - {high_thresh:.2f}')
print(f'High threshold : > {high_thresh:.2f}')


def map_stress_to_risk(score):
    if score < low_thresh:
        return 0  # Low
    elif score < high_thresh:
        return 1  # Medium
    else:
        return 2  # High


y_pred_final = [map_stress_to_risk(s) for s in xgb_pred]
y_test_final = [map_stress_to_risk(s) for s in y_te]

print('\n Classification Report (After XGBoost Regression) ')
print(classification_report(y_test_final, y_pred_final,
      target_names=['Low', 'Medium', 'High']))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test_final, y_pred_final), annot=True, fmt='d', cmap='Greens',
            xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.title('Confusion Matrix — Regression → Classification')
plt.show()

---
# 🏷️ PART B — CLASSIFICATION (Predict burnout_level)
---

In [ ]:
X_clf = df_clean.drop(columns=['stress_level', 'burnout_level'])
y_clf = df_clean['burnout_level']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler_c = MinMaxScaler()
X_tr_c_sc = scaler_c.fit_transform(X_tr_c)
X_te_c_sc = scaler_c.transform(X_te_c)

clf_records = []
print(f'Train: {X_tr_c_sc.shape}, Test: {X_te_c_sc.shape}')

In [ ]:
# B1 — Boosting Models
print(' B1: Boosting Models ')

gbc = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3)
gbc.fit(X_tr_c_sc, y_tr_c)
gbc_acc = accuracy_score(y_te_c, gbc.predict(X_te_c_sc))
clf_records.append({'Model': 'Gradient Boosting',
                   'Accuracy': round(gbc_acc, 4)})
print(f'GBC     : {gbc_acc:.4f}')

xgb_c = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8, eval_metric='logloss')
xgb_c.fit(X_tr_c_sc, y_tr_c)
xgb_acc = accuracy_score(y_te_c, xgb_c.predict(X_te_c_sc))
clf_records.append({'Model': 'XGBoost', 'Accuracy': round(xgb_acc, 4)})
print(f'XGBoost : {xgb_acc:.4f}')

cat = CatBoostClassifier(iterations=300, learning_rate=0.05,
                         depth=6, l2_leaf_reg=3.0, verbose=False)
cat.fit(X_tr_c_sc, y_tr_c)
cat_acc = accuracy_score(y_te_c, cat.predict(X_te_c_sc))
clf_records.append({'Model': 'CatBoost', 'Accuracy': round(cat_acc, 4)})
print(f'CatBoost: {cat_acc:.4f}')

In [ ]:
# B2 — SVM with 5-Fold Cross Validation
print(' B2: SVM + 5-Fold CV ')


def perf_eval(y_true, y_prob):
    yp = np.argmax(y_prob, axis=1)
    ACC = accuracy_score(y_true, yp)
    REC = recall_score(y_true, yp, average='macro', zero_division=0)
    PRE = precision_score(y_true, yp, average='macro', zero_division=0)
    MCC = matthews_corrcoef(y_true, yp)
    F1 = f1_score(y_true, yp, average='macro', zero_division=0)
    AUC = roc_auc_score(y_true, y_prob, multi_class='ovr')
    return [round(x, 4) for x in [ACC, REC, PRE, MCC, F1, AUC]]


svm_model = SVC(probability=True)
svm_model.fit(X_tr_c_sc, y_tr_c)

yp_tr_prob = cross_val_predict(
    svm_model, X_tr_c_sc, y_tr_c, cv=5, method='predict_proba')
yp_te_prob = svm_model.predict_proba(X_te_c_sc)

cols = ['ACC', 'Recall', 'Precision', 'MCC', 'F1', 'AUC']
tr_m = perf_eval(y_tr_c, yp_tr_prob)
te_m = perf_eval(y_te_c, yp_te_prob)

print(f'Metrics : {cols}')
print(f'Train   : {tr_m}')
print(f'Test    : {te_m}')
clf_records.append({'Model': 'SVM (5-Fold CV)', 'Accuracy': te_m[0]})

In [ ]:
# B3 — RandomizedSearchCV
print(' B3: RandomizedSearchCV ')

param_gb = {'n_estimators': np.arange(
    100, 500, 50), 'learning_rate': np.logspace(-3, -1, 20), 'max_depth': np.arange(3, 8)}
param_xgb = {'n_estimators': np.arange(100, 500, 50), 'learning_rate': np.logspace(
    -3, -1, 20), 'max_depth': np.arange(3, 10), 'subsample': np.linspace(0.6, 1.0, 5)}

rand_gbc = RandomizedSearchCV(GradientBoostingClassifier(
), param_gb, n_iter=20, cv=5, scoring='accuracy', n_jobs=-1, random_state=42, verbose=0)
rand_xgb = RandomizedSearchCV(XGBClassifier(eval_metric='logloss'), param_xgb,
                              n_iter=20, cv=5, scoring='accuracy', n_jobs=-1, random_state=42, verbose=0)

rand_gbc.fit(X_tr_c_sc, y_tr_c)
rand_xgb.fit(X_tr_c_sc, y_tr_c)

tuned_gbc_acc = accuracy_score(y_te_c, rand_gbc.predict(X_te_c_sc))
tuned_xgb_acc = accuracy_score(y_te_c, rand_xgb.predict(X_te_c_sc))

print(f'Best GBC params: {rand_gbc.best_params_}')
print(f'Best XGB params: {rand_xgb.best_params_}')
print(f'Tuned GBC Acc  : {tuned_gbc_acc:.4f}')
print(f'Tuned XGB Acc  : {tuned_xgb_acc:.4f}')

clf_records.append({'Model': 'Tuned GBC (RandomSearch)',
                   'Accuracy': round(tuned_gbc_acc, 4)})
clf_records.append({'Model': 'Tuned XGB (RandomSearch)',
                   'Accuracy': round(tuned_xgb_acc, 4)})

In [ ]:
# B4 — Ensemble Voting Classifier (GridSearchCV)
print(' B4: Ensemble Voting + GridSearchCV ')

eclf = VotingClassifier(
    estimators=[('lr', LogisticRegression(random_state=42)),
                ('et', ExtraTreesClassifier(random_state=42)),
                ('svm', SVC(probability=True, random_state=42))],
    voting='soft'
)
params = {'lr__C': [1.0, 10.0],
          'et__n_estimators': [50, 100], 'svm__C': [4, 8]}

best_eclf = GridSearchCV(eclf, param_grid=params, cv=5,
                         scoring='accuracy', n_jobs=-1, verbose=0)
best_eclf.fit(X_tr_c_sc, y_tr_c)

final_model = best_eclf.best_estimator_
final_model.fit(X_tr_c_sc, y_tr_c)

yp_tr_vc = cross_val_predict(
    final_model, X_tr_c_sc, y_tr_c, cv=5, method='predict_proba')
yp_te_vc = final_model.predict_proba(X_te_c_sc)

vc_tr = perf_eval(y_tr_c, yp_tr_vc)
vc_te = perf_eval(y_te_c, yp_te_vc)

print(f'Best Params: {best_eclf.best_params_}')
print(f'Train: ACC={vc_tr[0]}, F1={vc_tr[4]}, AUC={vc_tr[5]}')
print(f'Test : ACC={vc_te[0]}, F1={vc_te[4]}, AUC={vc_te[5]}')
clf_records.append(
    {'Model': 'Ensemble Voting (GridSearch)', 'Accuracy': vc_te[0]})

In [ ]:
# B5 — Pipeline: 5 Classic Models
print(' B5: Pipeline Models ')

X_raw = df_clean.drop(columns=['stress_level', 'burnout_level'])
y_raw = df_clean['burnout_level']
num_c = X_raw.select_dtypes(include=['float64', 'int64']).columns
cat_c = X_raw.select_dtypes(include=['object', 'category']).columns

pre = ColumnTransformer([
    ('num', StandardScaler(), num_c),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_c)
])

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

pipe_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'SVM (Pipeline)':      SVC(),
    'KNN':                 KNeighborsClassifier()
}

for name, model in pipe_models.items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    pipe.fit(X_tr_p, y_tr_p)
    acc = accuracy_score(y_te_p, pipe.predict(X_te_p))
    clf_records.append({'Model': name, 'Accuracy': round(acc, 4)})
    print(f'{name:25s}: {acc:.4f}')

##  Final Results — All Classification Models

In [ ]:
clf_df = pd.DataFrame(clf_records).sort_values(
    'Accuracy', ascending=False).reset_index(drop=True)
clf_df.index += 1
print(clf_df.to_string())

plt.figure(figsize=(14, 6))
colors = ['gold' if i == 0 else 'mediumseagreen' for i in range(len(clf_df))]
bars = plt.barh(clf_df['Model'], clf_df['Accuracy'], color=colors)
plt.xlabel('Accuracy')
plt.title(' Developer Burnout — All Classification Models Compared')
plt.xlim(0.5, 1.0)
for bar, val in zip(bars, clf_df['Accuracy']):
    plt.text(bar.get_width()+0.001, bar.get_y()+bar.get_height() /
             2, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

best = clf_df.iloc[0]
print(f'\n Best Model: {best["Model"]} — Accuracy: {best["Accuracy"]}')

---
# ✅ Summary — Developer Burnout Dataset

| Part | Task | Target | Models |
|------|------|--------|--------|
| A | Regression | stress_level | Linear, GB, XGBoost |
| B1 | Classification | burnout_level | GBC, XGBoost, CatBoost |
| B2 | SVM + 5-Fold CV | burnout_level | SVC |
| B3 | Hyperparameter Tuning | burnout_level | RandomizedSearchCV |
| B4 | Ensemble Voting | burnout_level | LR + ET + SVM (GridSearch) |
| B5 | Pipeline Models | burnout_level | LR, RF, DT, SVM, KNN |

## 🔑 Developer-Specific Features Used
- `bugs_per_day`, `commits_per_day` — productivity indicators
- `caffeine_intake` — health indicator
- `experience_years` — seniority
- `exercise_hours` — wellness
- `stress_level` — continuous burnout measure
---